In [0]:
bronze_count = spark.read.table("nyc_taxi_project.raw_data.bronze_taxi").count()
silver_count = spark.read.table("nyc_taxi_project.silver_data.silver_taxi").count()

print(f"Bronze rows:  {bronze_count:,}")
print(f"Silver rows:  {silver_count:,}")
print(f"Rows removed: {bronze_count - silver_count:,}")
print(f"Data retained: {round((silver_count/bronze_count)*100, 2)}%")

In [0]:
from pyspark.sql.functions import col, count, when

df_silver = spark.read.table("nyc_taxi_project.silver_data.silver_taxi")

null_counts = df_silver.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["pickup_datetime", "dropoff_datetime", "fare_amount", 
              "trip_distance", "passenger_count", "trip_duration_minutes"]
])

display(null_counts)

In [0]:
from pyspark.sql.functions import min as spark_min, max as spark_max, avg

display(df_silver.select(
    spark_min("fare_amount").alias("min_fare"),
    spark_max("fare_amount").alias("max_fare"),
    avg("fare_amount").alias("avg_fare"),
    spark_min("trip_distance").alias("min_distance"),
    spark_max("trip_distance").alias("max_distance"),
    avg("trip_distance").alias("avg_distance"),
    spark_min("trip_duration_minutes").alias("min_duration"),
    spark_max("trip_duration_minutes").alias("max_duration"),
    avg("trip_duration_minutes").alias("avg_duration"),
    spark_min("passenger_count").alias("min_passengers"),
    spark_max("passenger_count").alias("max_passengers"),
    avg("passenger_count").alias("avg_passengers")
))

In [0]:
total = df_silver.count()
distinct = df_silver.dropDuplicates().count()

print(f"Total rows:    {total:,}")
print(f"Distinct rows: {distinct:,}")
print(f"Duplicates:    {total - distinct:,}")

In [0]:
display(
    spark.sql("""
        SELECT 
            payment_type,
            COUNT(*) as total_trips,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
        FROM nyc_taxi_project.silver_data.silver_taxi
        GROUP BY payment_type
        ORDER BY total_trips DESC
    """)
)

In [0]:
print("=" * 50)
print("DATA QUALITY REPORT — NYC Taxi Project")
print("=" * 50)
print(f"Bronze Layer Rows : {bronze_count:,}")
print(f"Silver Layer Rows : {silver_count:,}")
print(f"Rows Cleaned      : {bronze_count - silver_count:,}")
print(f"Data Quality Score: {round((silver_count/bronze_count)*100, 2)}%")
print("Null values       : 0 (enforced via dropna)")
print("Duplicates        : 0 (enforced via dropDuplicates)")
print("Fare range        : $0.01 – $500")
print("Distance range    : 0.01 – 100 miles")
print("Duration range    : 1 – 180 minutes")
print("Coordinate bounds : NYC only")
print("=" * 50)